# SuperMimic × SUGAR 云端 Colab 全流程复现与 TensorBoard 可视化指南

> **[重要源码来源与未开源缺失声明]**
> 1. **SuperMimic 官方代码缺失声明**：**SuperMimic 原论文官方代码尚未开源（官方源码缺失）**。本工程中的 `sugar_rl.supermimic` 模块（参考本地非官方实现 `allmimic-main` 重构移植）属于**依据论文方法编写的非官方融合实现，并非原论文作者官方源码**。
> 2. **SUGAR 官方源码基础**：全身人-物交互仿真环境（`sugar_rl/tasks`）、机器人/物体资产与高层扩散生成器（`sugar_il`）均来自已核实的 **SUGAR 官方开源仓库**。
> 3. **验证边界说明**：核心数学、正向运动学、18 项特权观测切片硬断言与 32 步 BPTT 模块已通过本地 CUDA 9 项回归测试；首次在 Colab 运行完整长训练前，请先执行 **Step 5（20 轮实机冒烟测试 + TensorBoard 写入验证）**。

---

## 本 Notebook 在 Google Colab 上的三大核心特性

1. **专为 Google Colab 一键跑通优化**：
   - 自动通过 `uv` 创建独立的 **Python 3.11** 虚拟环境 `/content/.venv_fusion`（解决 Colab 默认 Python 3.12 内核无法直接安装 `isaacsim==5.1.0` `cp311` wheel 的问题）；
   - 自动适配 **Colab A100 / L4 / T4** 显卡（Stage 5 扩散训练自动检测硬件是否支持 `bf16`，在 T4 上自动无缝回退 `fp16`）；
   - 使用 SUGAR 官方 Google Drive 文件 ID（`gdown`）高速拉取 `descriptions.zip` 与 `data.zip`，并支持挂载你自己的 **Google Drive 实时增量备份权重与 TensorBoard 日志**，防止 Colab 断连丢失进度。
2. **全流程三阶段训练数据 100% 原生支持 TensorBoard 查看**：
   - **Stage 1 (`SuperMimic Refiner`)** → 自动写入 `outputs/<TASK>_<EXP>/logs/refiner/events.out.tfevents.*`（每 10 轮自动 `flush`）；
   - **Stage 3 (`SuperMimic Tracker`)** → 自动写入 `outputs/<TASK>_<EXP>/logs/tracker/events.out.tfevents.*`（含 Teacher BC 退火曲线）；
   - **Stage 5 (`Diffusion Generator`)** → `Accelerate` 自动写入 `outputs/<TASK>_<EXP>/logs/generator/tb/train_generator/events.out.tfevents.*`；
   - 支持 **Colab 内嵌 `%tensorboard` 交互式面板实时监控** + **Event 文件直接解析与 Matplotlib 离线曲线导出**（彻底避免浏览器拦截第三方 Cookie 导致 TensorBoard 白屏）。
3. **双阶段世界模型 BPTT 完整闭环**：从 Stage 1 Refiner → Stage 2 物理过滤 → Stage 3 Tracker → Stage 4 Zarr 数据转换 → Stage 5 Diffusion Generator → Stage 6 闭环推理 MP4 视频内嵌播放。

---

## 运行前准备工作：本地代码上传 GitHub → Colab 拉取复现

请按以下 **3 个准备步骤** 完成代码上云（后续每次本地改代码，只需 `git push`，在 Colab 重复运行 Step 1 即可自动拉取最新代码）：

### 准备 1：在 GitHub 创建仓库
1. 登录 GitHub，点击右上角 **`+` → `New repository`**，创建一个空仓库（例如命名为 `sugar-supermimic-fusion`，公开或私有均可，**不要勾选** `Add a README file` 或 `.gitignore`，保持为空仓库）。
2. 若使用**私有仓库**，请提前生成一个只读拉取权限的 Personal Access Token（GitHub 右上角头像 → `Settings` → `Developer settings` → `Personal access tokens` → `Fine-grained tokens` 或 `Tokens (classic)`，勾选 `repo` 权限）。

### 准备 2：在 Windows 本地 PowerShell 将 `fusion` 推送到 GitHub
在本地打开 **Windows PowerShell**，依次执行以下命令（注意将 `<你的GitHub用户名>` 和 `<你的仓库名>` 替换为真实名称）：

```powershell

cd D:\robot\rework\fusion
git config --global user.name "misaka142"
git config --global user.email "1421061705@qq.com"
git commit -m "feat: add real-time Google Drive sync and mid-stage auto-resume"
git push -u origin main

cd D:\robot\rework\fusion
git init
git branch -M main
git status -s
git add .git push -u origin main --force
git commit -m "feat: SuperMimic x SUGAR fusion project"
git remote add origin https://github.com/misaka142/fusion.git
git push -u origin main
```
- **命令作用**：初始化本地 Git 仓库，借助项目自带的 `.gitignore` 自动过滤 `data/`、`descriptions/`、`outputs/` 等大体积资产，将纯代码与本 Notebook 推送至 GitHub。
- **关键参数含义**：
  - `git branch -M main`：将默认分支统一命名为 `main`（与 Colab Step 1 默认配置一致）；
  - `git status -s`：推送前检查待提交文件清单，确认只包含源码与脚本（几 MB 以内），无任何大体积数据集；
  - `git push -u origin main`：将代码推送至 GitHub 并将 `origin/main` 设为默认上游分支。
- **预期结果**：终端输出 `Branch 'main' set up to track remote branch 'main' from 'origin'`，刷新 GitHub 网页即可看到 `SuperMimic_SUGAR_Repro.ipynb`、`source/`、`scripts/` 等全部代码。

### 准备 3：在 Google Colab 打开本 Notebook 并切换 GPU 运行时
1. 打开 [Google Colab](https://colab.research.google.com/)，选择 **文件 (`File`) → 打开笔记本 (`Open notebook`) → GitHub**，输入你的仓库名打开 `SuperMimic_SUGAR_Repro.ipynb`（若为私有仓库，先勾选「包含私有仓库」或直接本地上传本 `.ipynb` 文件）。
2. 点击顶部菜单 **代码执行程序 (`Runtime`) → 更改运行时类型 (`Change runtime type`)**，硬件加速器选择 **GPU**（推荐 **A100** 或 **L4**；免费版 **T4** 也可跑通，但建议用 `fast_verify` 模式）。
3. 进入下方 **Step 0** 开始依次执行！

---
## Step 0：Colab GPU 型号、显存余量与磁盘空间自检

- **本步作用**：确认当前 Colab 实例已开启 NVIDIA GPU 硬件加速（A100 / L4 / T4），根据显存容量给出并行环境数 `NUM_ENVS` 推荐值，并检查 `/content` 磁盘可用容量（完整流水线建议可用空间 `>= 30 GB`）。
- **关键参数**：无需手动修改，自动探测 `google.colab`、`nvidia-smi` 与磁盘余量。
- **预期结果**：打印 GPU 型号、显存大小、推荐 `NUM_ENVS` 与可用磁盘空间。

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print("=== 运行平台检测 ===")
print(f"OS / Python : {platform.platform()} | Host Python {sys.version.split()[0]}")
print(f"运行环境    : {'Google Colab 云端 GPU 实例' if IS_COLAB else 'Linux 本地 GPU 环境'}")
print(f"当前工作目录: {Path.cwd()}")

gpu_res = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
)
if gpu_res.returncode != 0 or not gpu_res.stdout.strip():
    raise RuntimeError(
        "未检测到 NVIDIA GPU！请在 Colab 顶部菜单点击『代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)』，选择 GPU (推荐 A100 或 L4)。"
    )

print("\n=== NVIDIA GPU 状态 ===")
GPU_VRAM_GB = 16.0
for line in gpu_res.stdout.strip().splitlines():
    parts = [p.strip() for p in line.split(",")]
    idx, name, mem_used, mem_tot, util = parts[0], parts[1], float(parts[2]), float(parts[3]), parts[4]
    GPU_VRAM_GB = max(GPU_VRAM_GB, mem_tot / 1024.0)
    print(f"  GPU {idx}: {name} | 已用 {mem_used:.0f} MB / 总计 {mem_tot:.0f} MB ({GPU_VRAM_GB:.1f} GB) | 利用率 {util}%")

if GPU_VRAM_GB >= 38:
    SUGGESTED_FULL_ENVS = 4096
elif GPU_VRAM_GB >= 20:
    SUGGESTED_FULL_ENVS = 2048
else:
    SUGGESTED_FULL_ENVS = 512
print(f"  -> 根据当前 {GPU_VRAM_GB:.1f} GB 显存，完整训练推荐并行环境数 NUM_ENVS = {SUGGESTED_FULL_ENVS}")

total_b, used_b, free_b = shutil.disk_usage(Path.cwd())
print(f"\n=== 磁盘空间余量 ===")
print(f"总容量: {total_b / (1024**3):.1f} GB | 已用: {used_b / (1024**3):.1f} GB | 剩余可用: {free_b / (1024**3):.1f} GB")
if free_b / (1024**3) < 25.0:
    print("[警告] 剩余磁盘空间低于 25 GB，建议在 Colab 选择高磁盘配额或开启下方 Google Drive 备份。")

---
## Step 1：挂载 Google Drive（可选备份）并从 GitHub 克隆/同步 `fusion` 代码

- **本步作用**：
  1. 在 Colab 上默认挂载 Google Drive 至 `/content/drive`（创建持久化目录 `MyDrive/SuperMimic_SUGAR_Backup`，自动备份模型检查点 `ckpts/` 与 **TensorBoard 日志 `logs/`**，防止 Colab 断连丢失进度）；
  2. 从你在准备工作推送好的 **GitHub 仓库**将代码克隆到 `/content/fusion`（若目录已存在则自动执行 `git pull --ff-only` 同步最新提交；私有仓库通过 `getpass` 隐藏输入 Token 并在克隆后立即从 `.git/config` 抹除明文 Token）；
  3. 初始化实时日志流式执行器 `run_stream()`（含 Isaac Sim 5.1 退出码 `139` 安全容错）。
- **关键参数**（运行前请填写下方 `GITHUB_REPO`）：
  - `GITHUB_REPO`：**必填**，填写你在准备阶段创建的 GitHub 仓库，格式为 `"你的GitHub用户名/你的仓库名"`（也可直接粘贴完整的 `https://github.com/...` 仓库地址）；
  - `GITHUB_BRANCH`：分支名，默认为 `"main"`；
  - `REPO_IS_PRIVATE`：若为私有仓库设为 `True`（运行后会弹出隐藏输入框要求输入 GitHub PAT），公开仓库保持 `False` 即可免密克隆；
  - `USE_DRIVE_BACKUP`：Colab 下是否挂载 Google Drive 自动同步权重和 TensorBoard 日志（默认 `True`）。
- **预期结果**：成功克隆或更新至 `/content/fusion`，输出 `项目根目录已锁定: /content/fusion`。

In [ ]:
import getpass
import threading
import time

# ==================== 运行前请填写你的 GitHub 仓库信息 ====================
GITHUB_REPO = "misaka142/fusion"  # 默认指向你的 GitHub 仓库；若更换仓库请修改此处
GITHUB_BRANCH = "main"         # 与本地 `git branch -M main` 保持一致
REPO_IS_PRIVATE = False        # 私有仓库请改为 True（运行后会提示隐藏输入 GitHub PAT）

USE_DRIVE_BACKUP = IS_COLAB    # Colab 推荐开启：训练期间实时把模型检查点和 TensorBoard 日志增量同步到 Google Drive
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/SuperMimic_SUGAR_Backup"
DRIVE_SYNC_INTERVAL_SEC = 90   # 训练期间后台自动同步至 Google Drive 的时间间隔（秒），且每次存盘瞬间立即触发同步
# ========================================================================

BACKUP_DIR = None
if IS_COLAB and USE_DRIVE_BACKUP:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    BACKUP_DIR = Path(DRIVE_BACKUP_ROOT)
    (BACKUP_DIR / "wheels_and_zips").mkdir(parents=True, exist_ok=True)
    (BACKUP_DIR / "outputs").mkdir(parents=True, exist_ok=True)
    print(f"[Google Drive] 已挂载持久化备份目录: {BACKUP_DIR}")


def _normalize_repo_slug(repo_cfg: str) -> str:
    slug = repo_cfg.strip().rstrip("/")
    if slug.startswith("https://github.com/"):
        slug = slug[len("https://github.com/"):]
    elif slug.startswith("http://github.com/"):
        slug = slug[len("http://github.com/"):]
    if slug.endswith(".git"):
        slug = slug[:-4]
    return slug


def resolve_repo_dir() -> Path:
    # 1. 若 Colab 或本地目录已经克隆好项目，优先复用（Colab 下支持自动 git pull 更新）
    local_candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/fusion"),
    ]
    for c in local_candidates:
        if (c / "source" / "sugar_rl" / "sugar_rl" / "supermimic" / "trainer.py").is_file():
            if IS_COLAB and (c / ".git").is_dir() and not REPO_IS_PRIVATE:
                print(f"[Git Sync] 检测到已有仓库 {c}，尝试拉取最新提交...")
                subprocess.run(["git", "-C", str(c), "pull", "--ff-only"], check=False)
            return c.resolve()

    # 2. 从 GitHub 克隆到 Colab `/content/fusion`
    slug = _normalize_repo_slug(GITHUB_REPO)
    if not slug or "/" not in slug:
        raise ValueError(
            "请先在上方配置区填写 `GITHUB_REPO = '你的GitHub用户名/仓库名'` 再运行本单元格！"
        )

    target = Path("/content/fusion")
    pat = ""
    if REPO_IS_PRIVATE:
        pat = getpass.getpass("请输入 GitHub Personal Access Token (隐藏输入，不会回显): ").strip()
    clone_url = f"https://{pat}@github.com/{slug}.git" if pat else f"https://github.com/{slug}.git"

    print(f"[Git Clone] 正在从 GitHub ({slug}@{GITHUB_BRANCH}) 下载代码至 {target} ...")
    res = subprocess.run(
        ["git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1", clone_url, str(target)],
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        safe_err = res.stderr.replace(pat, "***") if pat else res.stderr
        raise RuntimeError(
            f"Git 克隆失败！请检查仓库名 `{slug}`、分支 `{GITHUB_BRANCH}` 是否正确"
            f"（若为私有仓库，请将 `REPO_IS_PRIVATE = True` 并确认 Token 具有 repo 权限）。\n原始报错:\n{safe_err}"
        )
    if pat:
        # 立即清除 .git/config 中的明文 Token
        subprocess.run(["git", "-C", str(target), "remote", "set-url", "origin", f"https://github.com/{slug}.git"], check=True)
    return target.resolve()


REPO_DIR = resolve_repo_dir()
os.chdir(REPO_DIR)


def sync_to_drive(rel_path: str, quiet: bool = False) -> bool:
    """Colab 模式下将权重或 TensorBoard 日志安全增量同步到 Google Drive（自动过滤未写完的 .tmp 临时文件）。"""
    if BACKUP_DIR is None:
        return False
    src = REPO_DIR / rel_path
    dst = BACKUP_DIR / rel_path
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        cmd = ["rsync", "-a", "--exclude=*.tmp", "--exclude=*_sample_attn_maps", f"{src}/", f"{dst}/"]
    else:
        cmd = ["rsync", "-a", "--exclude=*.tmp", str(src), str(dst)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[Drive Sync WARN] 同步 {rel_path} 返回码 {res.returncode}: {res.stderr.strip()}", flush=True)
        return False
    if not quiet:
        print(f"[Drive Sync] 已同步至 Google Drive: {rel_path}", flush=True)
    return True


def run_stream(
    cmd: list[str],
    cwd: Path = REPO_DIR,
    env: dict | None = None,
    log_file: Path | None = None,
    allow_isaac_139: bool = False,
    expected_output: Path | None = None,
    on_line=None,
) -> int:
    """实时流式打印子进程输出（含 TensorBoard 路径提示），并安全处理 Isaac Sim 退出码 139/-11。"""
    merged_env = os.environ.copy()
    merged_env["PYTHONUNBUFFERED"] = "1"
    merged_env["OMNI_KIT_ACCEPT_EULA"] = "YES"
    merged_env["ACCEPT_EULA"] = "Y"
    merged_env["PRIVACY_CONSENT"] = "Y"
    if env:
        merged_env.update(env)

    f_log = None
    if log_file is not None:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        f_log = open(log_file, "a", encoding="utf-8")
        f_log.write(f"\n=== Launch @ {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n")
        f_log.flush()

    print(f"\n[CMD] ({cwd}) $ {' '.join(str(x) for x in cmd)}")
    t0 = time.time()
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd),
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            if f_log is not None:
                f_log.write(line)
                f_log.flush()
            if on_line is not None:
                on_line(line)
        rc = proc.wait()
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
        if f_log is not None:
            f_log.close()

    elapsed = time.time() - t0
    if rc != 0:
        if allow_isaac_139 and rc in (139, -11):
            if expected_output is None or expected_output.exists():
                print(f"\n[提示] Isaac Sim 退出析构返回码 {rc}，目标产物已核实完好 (耗时 {elapsed:.1f}s)。")
                return 0
        raise RuntimeError(f"命令执行失败 (返回码 {rc}, 耗时 {elapsed:.1f}s): {' '.join(str(x) for x in cmd)}")

    print(f"[完成] 耗时 {elapsed:.1f}s")
    return 0


def run_training_stage(
    cmd: list[str],
    env: dict | None = None,
    log_file: Path | None = None,
    allow_isaac_139: bool = False,
    expected_output: Path | None = None,
    sync_rel_paths: list[str] | None = None,
    sync_interval_sec: float = DRIVE_SYNC_INTERVAL_SEC,
) -> int:
    """流式运行训练阶段，并在训练进行中实时将最新检查点与 TensorBoard 日志增量同步至 Google Drive。"""
    if BACKUP_DIR is None or not sync_rel_paths:
        return run_stream(
            cmd=cmd,
            cwd=REPO_DIR,
            env=env,
            log_file=log_file,
            allow_isaac_139=allow_isaac_139,
            expected_output=expected_output,
        )

    stop_evt = threading.Event()
    trigger_evt = threading.Event()

    def _bg_sync_loop() -> None:
        while not stop_evt.is_set():
            triggered = trigger_evt.wait(timeout=sync_interval_sec)
            trigger_evt.clear()
            if stop_evt.is_set():
                break
            synced_any = False
            for rel_p in sync_rel_paths:
                synced_any = sync_to_drive(rel_p, quiet=True) or synced_any
            if triggered and synced_any:
                print("[Drive Auto-Sync] 已实时备份最新检查点与日志 -> Google Drive", flush=True)

    def _handle_line(line: str) -> None:
        if "Saved checkpoint ->" in line:
            trigger_evt.set()

    bg_thread = threading.Thread(target=_bg_sync_loop, daemon=True)
    bg_thread.start()
    try:
        return run_stream(
            cmd=cmd,
            cwd=REPO_DIR,
            env=env,
            log_file=log_file,
            allow_isaac_139=allow_isaac_139,
            expected_output=expected_output,
            on_line=_handle_line,
        )
    finally:
        stop_evt.set()
        trigger_evt.set()
        bg_thread.join(timeout=30.0)
        for rel_p in sync_rel_paths:
            sync_to_drive(rel_p, quiet=True)


print(f"项目根目录已锁定: {REPO_DIR}")
assert (REPO_DIR / "scripts" / "supermimic" / "train.py").is_file(), "缺少 scripts/supermimic/train.py"
print("核心文件结构校验通过！")

---
## Step 2：在 Colab 构建 Python 3.11 + Isaac Sim 5.1.0 + Isaac Lab v2.3.0 + TensorBoard 环境

- **本步作用**：
  1. 在 Colab 上利用高速包管理器 `uv` 创建独立的 **Python 3.11** 虚拟环境 `/content/.venv_fusion`（首次安装约需 12~16 分钟，同一 Colab 会话内重复运行自动跳过）；
  2. 安装离屏渲染系统库（`libegl1`、`ffmpeg` 等，支持 Headless 录制 `.mp4` 视频）；
  3. 严格按 SUGAR 官方版本栈安装 `isaacsim[all,extscache]==5.1.0`、`flatdict==4.0.1`、`IsaacLab v2.3.0`；
  4. **一次性装齐 `tensorboard` 以及 Stage 4/5 扩散策略所需的全部依赖**（`accelerate`, `diffusers`, `zarr<3.0`, `numcodecs<0.16`, `hydra-core`, `einops`, `dill`, `timm`, `gdown`），确保从 Stage 1 到 Stage 6 及 TensorBoard 日志记录全程零缺包中断！
- **关键参数**：`EXISTING_PYTHON_BIN` 留空即可自动构建或复用 `/content/.venv_fusion`。
- **预期结果**：打印 `[OK] Loaded supermimic` 与 `[OK] TensorBoard SummaryWriter verified!`。

In [ ]:
EXISTING_PYTHON_BIN = ""  # Colab 保持留空自动构建；若在已有环境的 Linux 服务器可指定 python 路径


def has_module(py_bin: str, mod_name: str) -> bool:
    r = subprocess.run([py_bin, "-c", f"import {mod_name}"], capture_output=True, text=True)
    return r.returncode == 0


def setup_or_locate_python_env() -> str:
    probe_list = [p for p in [EXISTING_PYTHON_BIN, "/content/.venv_fusion/bin/python", sys.executable] if p and Path(p).is_file()]
    if not IS_COLAB:
        for conda_root in [Path.home() / "miniconda3" / "envs", Path.home() / "anaconda3" / "envs", Path("/opt/conda/envs")]:
            if conda_root.is_dir():
                for env_dir in sorted(conda_root.iterdir()):
                    cand = env_dir / "bin" / "python"
                    if cand.is_file():
                        probe_list.append(str(cand))

    for py_cand in probe_list:
        if has_module(py_cand, "isaaclab"):
            print(f"[环境就绪] 检测到已安装 IsaacLab 的 Python 3.11 环境: {py_cand}")
            return py_cand

    venv_dir = Path("/content/.venv_fusion") if IS_COLAB else (REPO_DIR / ".venv_fusion")
    py_bin = str(venv_dir / "bin" / "python")
    isaaclab_dir = Path("/content/IsaacLab") if IS_COLAB else (REPO_DIR.parent / "IsaacLab_v2_3_0")

    if IS_COLAB:
        print("[1/5] 安装 Colab 离屏相机渲染与工具依赖 (libegl1, ffmpeg, rsync, unzip)...")
        run_stream(["apt-get", "update", "-qq"])
        run_stream([
            "apt-get", "install", "-y", "-qq",
            "libegl1", "libgl1", "libglu1-mesa", "libvulkan1", "libosmesa6-dev", "libgomp1", "libxi6", "libxrandr2", "libxcursor1", "libxinerama1",
            "ffmpeg", "rsync", "unzip",
        ])

    if not Path(py_bin).is_file():
        print("[2/5] 安装 uv 并创建独立 Python 3.11 虚拟环境...")
        run_stream([sys.executable, "-m", "pip", "install", "-q", "uv", "tensorboard", "gdown"])
        run_stream(["uv", "venv", "--python", "3.11", "--seed", str(venv_dir)])

    if not has_module(py_bin, "isaacsim"):
        print("[3/5] 安装 PyTorch 2.7.0+cu128 与 Isaac Sim 5.1.0 (预编译大包，请耐心等待约 8~10 分钟)...")
        run_stream([
            "uv", "pip", "install", "--python", py_bin,
            "torch==2.7.0", "torchvision==0.22.0",
            "--index-url", "https://download.pytorch.org/whl/cu128",
        ])
        run_stream([
            "uv", "pip", "install", "--python", py_bin,
            "isaacsim[all,extscache]==5.1.0",
            "--extra-index-url", "https://pypi.nvidia.com",
        ])

    if not has_module(py_bin, "isaaclab"):
        print("[4/5] 安装 setuptools/wheel、flatdict==4.0.1 与 Isaac Lab v2.3.0...")
        run_stream(["uv", "pip", "install", "--python", py_bin, "setuptools<75", "wheel"])
        run_stream(["uv", "pip", "install", "--python", py_bin, "--no-build-isolation", "flatdict==4.0.1"])
        if not isaaclab_dir.exists():
            run_stream(["git", "clone", "--branch", "v2.3.0", "--depth", "1", "https://github.com/isaac-sim/IsaacLab.git", str(isaaclab_dir)])
        env_with_venv = {"PATH": f"{venv_dir / 'bin'}:{os.environ.get('PATH', '')}", "VIRTUAL_ENV": str(venv_dir)}
        run_stream(["bash", str(isaaclab_dir / "isaaclab.sh"), "--install", "rsl_rl"], cwd=isaaclab_dir, env=env_with_venv)

    print("[5/5] 安装 TensorBoard 与 Stage 4/5 扩散策略依赖包 (zarr, diffusers, accelerate 等)...")
    run_stream([
        "uv", "pip", "install", "--python", py_bin,
        "tensordict>=0.7,<0.9", "tensorboard", "packaging", "accelerate", "diffusers",
        "zarr>=2.16,<3.0", "numcodecs<0.16", "hydra-core", "omegaconf",
        "einops", "dill", "scipy", "timm", "matplotlib", "tqdm", "gdown",
        "argcomplete", "prettytable", "psutil", "mujoco", "imageio[ffmpeg]", "trimesh", "scipy",
    ])
    return py_bin


PYTHON_BIN = setup_or_locate_python_env()

# 确保在复用已有环境时，训练解释器与宿主 Notebook 内核同样具备 tensorboard
if IS_COLAB and not has_module(PYTHON_BIN, "tensorboard"):
    run_stream([PYTHON_BIN, "-m", "pip", "install", "-q", "tensorboard", "packaging"])

INSTALL_EDITABLE = IS_COLAB
if INSTALL_EDITABLE:
    run_stream([PYTHON_BIN, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR / "source" / "sugar_rl")])
    run_stream([PYTHON_BIN, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR / "source" / "sugar_il")])

EXTRA_PYTHONPATH = f"{REPO_DIR / 'source' / 'sugar_rl'}:{REPO_DIR / 'source' / 'sugar_il'}"
BASE_ENV = {
    "PYTHONPATH": f"{EXTRA_PYTHONPATH}:{os.environ.get('PYTHONPATH', '')}".rstrip(":"),
    "HYDRA_FULL_ERROR": "1",
}

run_stream(
    [
        PYTHON_BIN,
        "-c",
        (
            "import sugar_rl.supermimic as sm, sugar_il; "
            "from torch.utils.tensorboard import SummaryWriter; "
            "print('[OK] Loaded supermimic from:', sm.__file__); "
            "print('[OK] Loaded sugar_il from:', sugar_il.__file__); "
            "print('[OK] TensorBoard SummaryWriter verified!')"
        ),
    ],
    env=BASE_ENV,
)

---
## Step 3：高速拉取官方 URDF 机器人描述、物体资产与 6 大任务参考数据集

- **本步作用**：使用 SUGAR 官方仓库发布的 Google Drive 文件 ID（通过 `gdown` 在 Colab 云端机房高速内网下载，并自动缓存到你挂载的 Google Drive 中避免下次重下）：
  1. `descriptions.zip`（~50 MB）：Unitree G1 29-DoF 带橡胶手 URDF 模型与交互物体网格；
  2. `data.zip`（~400 MB）：全部 6 个人-物交互任务（`CarryBox`, `PickBottle`, `StandBottle`, `PushBox`, `KickBox`, `SitChair`）参考动作集；
  3. `demo_ckpts.zip`（~250 MB，可选）：官方预训练权重，可用于交叉对比评估。
- **关键参数**：`DOWNLOAD_DEMO_CKPTS = True`（如不需要官方演示权重可改为 `False` 加快下载）。
- **预期结果**：确认 `g1_29dof_rev_1_0_with_rubber_hand.urdf`、`data/CarryBox` 与基于 `OmniRetarget_Dataset` 转换的 `data/OmniCarryBox`（含 `obj_aligned.usda` 凸包资产）完好就绪。

In [ ]:
DOWNLOAD_DEMO_CKPTS = True

# SUGAR 官方 README 核实的 Google Drive 文件 ID
OFFICIAL_GDRIVE_ASSETS = [
    ("descriptions", "descriptions.zip", "1wXNAjNMrfV0e-d2pQ6m9dm4xrG5lSoyD", REPO_DIR / "descriptions" / "robots" / "g1" / "g1_29dof_rev_1_0_with_rubber_hand.urdf"),
    ("data", "data.zip", "1AIJWqS5rFGl5u2Qq6jCCTHKdh51SX2Sc", REPO_DIR / "data" / "CarryBox"),
]
if DOWNLOAD_DEMO_CKPTS:
    OFFICIAL_GDRIVE_ASSETS.append(("demo_ckpts", "demo_ckpts.zip", "1Uc2SPPVvTboEgw4Scyuz3TmzNKDg-dx-", REPO_DIR / "demo_ckpts"))

SIBLING_SUGAR = REPO_DIR.parent / "SUGAR-main"
cache_zip_dir = (BACKUP_DIR / "wheels_and_zips") if BACKUP_DIR is not None else (REPO_DIR / ".asset_cache")
cache_zip_dir.mkdir(parents=True, exist_ok=True)

for folder_name, zip_name, gdrive_id, check_target in OFFICIAL_GDRIVE_ASSETS:
    target_dir = REPO_DIR / folder_name
    if check_target.exists():
        print(f"[已就绪] {folder_name}/ 已存在 ({check_target})")
        continue

    sibling_dir = SIBLING_SUGAR / folder_name
    if sibling_dir.exists() and not target_dir.exists():
        os.symlink(sibling_dir.resolve(), target_dir)
        print(f"[软链复用] {target_dir} -> {sibling_dir.resolve()}")
        continue

    zip_path = cache_zip_dir / zip_name
    if not zip_path.is_file():
        print(f"[下载中] {zip_name} (Google Drive ID: {gdrive_id})...")
        run_stream([PYTHON_BIN, "-m", "gdown", gdrive_id, "-O", str(zip_path)])

    print(f"[解压中] {zip_path} -> {REPO_DIR}")
    run_stream(["unzip", "-q", "-o", str(zip_path), "-d", str(REPO_DIR)])

urdf_path = REPO_DIR / "descriptions" / "robots" / "g1" / "g1_29dof_rev_1_0_with_rubber_hand.urdf"
assert urdf_path.is_file(), f"未找到关键 URDF 文件: {urdf_path}"
carrybox_subdirs = [p for p in (REPO_DIR / "data" / "CarryBox").iterdir() if p.is_dir()]
print(f"\n[校验通过] G1 URDF 完好 | data/CarryBox 包含 {len(carrybox_subdirs)} 组参考动作轨迹。")

# 自动准备 CarryBox 专用高质量 OmniRetarget 数据集 (`data/OmniCarryBox`)
OMNI_CARRYBOX_DIR = REPO_DIR / "data" / "OmniCarryBox"
import hashlib
OMNI_CONV_SHA = hashlib.sha256((REPO_DIR / "scripts" / "sugar_rl" / "convert_omniretarget_to_sugar.py").read_bytes()).hexdigest()[:8]
OMNI_SUBSET = "pick_carry_place_original"  # 先跑通 86 条原版高保真搬运动作；后期扩充可改为 pick_carry_place_all
OMNI_CACHE_ZIP = cache_zip_dir / f"OmniCarryBox_{OMNI_SUBSET}_{OMNI_CONV_SHA}.zip"

if (OMNI_CARRYBOX_DIR / "obj_aligned.usda").is_file() and len(list(OMNI_CARRYBOX_DIR.glob("data_*"))) > 0:
    print(f"[已就绪] OmniCarryBox 已存在 ({len(list(OMNI_CARRYBOX_DIR.glob('data_*')))} 条轨迹)")
elif OMNI_CACHE_ZIP.is_file():
    print(f"[Drive 缓存命中] 解压预转换数据集 {OMNI_CACHE_ZIP} -> {OMNI_CARRYBOX_DIR}")
    run_stream(["unzip", "-q", "-o", str(OMNI_CACHE_ZIP), "-d", str(REPO_DIR / "data")])
else:
    print(f"[OmniRetarget] 正在从 HuggingFace 下载并转换为 SUGAR 50Hz + USDA 凸包资产 ({OMNI_SUBSET})...")
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/sugar_rl/download_omniretarget.py",
            "--out_dir", "data_omniretarget_raw",
            "--cache_dir", str(cache_zip_dir),
        ],
        env=BASE_ENV,
    )
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/sugar_rl/convert_omniretarget_to_sugar.py",
            "--raw_dir", "data_omniretarget_raw",
            "--out_dir", str(OMNI_CARRYBOX_DIR),
            "--subset", OMNI_SUBSET,
        ],
        env=BASE_ENV,
    )
    if BACKUP_DIR is not None:
        import zipfile
        with zipfile.ZipFile(OMNI_CACHE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
            for p in sorted(OMNI_CARRYBOX_DIR.rglob("*")):
                if p.is_file():
                    zf.write(p, p.relative_to(REPO_DIR / "data"))
        print(f"[Drive Sync] 已缓存转换好的 OmniCarryBox 数据集至 -> {OMNI_CACHE_ZIP}")

assert (OMNI_CARRYBOX_DIR / "obj_aligned.usda").is_file(), "缺少 OmniCarryBox/obj_aligned.usda!"
assert (OMNI_CARRYBOX_DIR / "box_spec.json").is_file(), "缺少 OmniCarryBox/box_spec.json!"
print(f"[校验通过] OmniCarryBox ({len(list(OMNI_CARRYBOX_DIR.glob('data_*')))} 组轨迹) + obj_aligned.usda 就绪！")


---
## Step 4：运行 SuperMimic × SUGAR 14 项核心算法、编译等价与零同步 GPU 单元测试（秒级验证）

- **本步作用**：在启动 Isaac Sim 物理引擎前，用纯 PyTorch 在 Colab GPU 上快速验证旋转零角安全梯度、URDF 正向运动学与刚体局部 COM 力臂速度、890/510 维观测构建与 18 项切片硬断言、6 大任务分阶段奖励（含 `feet_height` 与 `motion_obj_pos_place`）、世界模型 `SE(2)` 不变性与 `delta_rms` 输出归一化、零主机同步回放采样、32 步 BPTT 整步编译梯度等价性以及 **RL/Generator 原子存盘与自动续训**。
- **关键参数**：运行 `tests/test_supermimic_fusion.py`。
- **预期结果**：14 项测试全部 `[PASS]`，末尾打印 `ALL 14 SUPERMIMIC x SUGAR TESTS PASSED SUCCESSFULLY!`。

In [ ]:
run_stream([PYTHON_BIN, str(REPO_DIR / "tests" / "test_supermimic_fusion.py")], env=BASE_ENV)

---
## Step 5：Isaac Sim 实机 20 轮冒烟测试 + TensorBoard 事件文件验证

- **本步作用**：
  1. 启动真实无头 Isaac Sim 5.1 物理引擎运行 20 轮 `SuperMimic Refiner` 短训练；
  2. 触发启动时的 **18 项特权观测切片硬断言**（`[SuperMimic Bridge] Privileged obs alignment verified ...`）；
  3. **当场核验 TensorBoard 事件文件 `events.out.tfevents.*` 是否已真实生成并包含完整的世界模型、Critic 与 32 步想象回报指标**，让你在正式长训练前 100% 放心数据可被 TensorBoard 读取！
- **关键参数**：`--num_envs 64 --max_iterations 20`。
- **预期结果**：产出 `model_19.pt` 并打印已写入 TensorBoard 的全部标量 Tag 清单。

In [ ]:
SMOKE_DIR = REPO_DIR / "outputs" / "smoke_test_refiner"
SMOKE_CKPT = SMOKE_DIR / "model_19.pt"

run_training_stage(
    [
        PYTHON_BIN, "-u", "scripts/supermimic/train.py",
        "--task", "Sugar-G129dof-CarryBox-Refiner",
        "--num_envs", "64",
        "--max_iterations", "20",
        "--save_interval", "10",
        "--motion_folder", "data/OmniCarryBox",
        "--log_dir", str(SMOKE_DIR),
        "--compile_bptt",
        "--compile_mode", "default",
        "--headless",
    ],
    env=BASE_ENV,
    log_file=SMOKE_DIR / "smoke_train.log",
    allow_isaac_139=True,
    expected_output=SMOKE_CKPT,
)

tfevents = sorted(SMOKE_DIR.glob("events.out.tfevents.*"))
assert SMOKE_CKPT.is_file(), f"冒烟测试未找到预期权重文件: {SMOKE_CKPT}"
assert tfevents, f"未在 {SMOKE_DIR} 下检测到 TensorBoard 事件文件！"
print(f"\n[冒烟测试与 TensorBoard 写入校验全部通过！]")
print(f"  权重文件       : {SMOKE_CKPT} ({SMOKE_CKPT.stat().st_size / (1024**2):.2f} MB)")
smoke_txt = (SMOKE_DIR / "smoke_train.log").read_text(encoding="utf-8", errors="ignore")
assert "[Dataset Box Override]" in smoke_txt, "冒烟日志未检测到 [Dataset Box Override]，请确认加载了 obj_aligned.usda!"
print(f"  TensorBoard日志: {tfevents[-1]} ({tfevents[-1].stat().st_size / 1024:.1f} KB)")
print("  [已核实] Stage C 冒烟日志已加载 OmniCarryBox obj_aligned.usda 凸包资产！")

---
## Step 6：配置完整 5 阶段训练流水线 & 启动实时内嵌 TensorBoard 面板

### 6.0 全局参数配置与实时 TensorBoard 启动说明

下面单元格统一配置任务与运行模式，并确定统一的 TensorBoard 根目录 `TB_LOGDIR = outputs/<TASK>_<EXP>/logs`。该目录下会自动汇聚**全部三个训练阶段**的子目录曲线：
- `refiner/`：Stage 6.1 `SuperMimic Refiner`（真实奖励 `collect/*`、世界模型误差 `wm/*`、32 步想象回报 `policy/*`、价值损失 `critic/loss`）；
- `tracker/`：Stage 6.3 `SuperMimic Tracker`（同上，外加冻结 Teacher 蒸馏损失 `policy/bc_loss` 与退火权重 `policy/bc_coef`）；
- `generator/tb/train_generator/`：Stage 6.5 `Diffusion Generator`（扩散损失 `train_loss`、动作重构误差 `train_action_mse_error`、学习率 `lr` 等）。

**运行模式选择**：
- `RUN_MODE = "fast_verify"`（首次跑通全流程推荐）：`RL_ITERS = 301`, `IL_EPOCHS = 101`。注：对于 `CarryBox` 等复杂任务，若 `301` 轮后 Stage 6.2 Rollout 过滤为 `0` 条，可将 `RL_ITERS` 调高至 `1000~1500`、执行 `REFINER_CKPT.unlink(missing_ok=True)` 后重跑 Stage 6.1，即可从第 `301` 轮自动接力续训；
- `RUN_MODE = "full"`（正式论文级精度）：`RL_ITERS = 5001`, `IL_EPOCHS = 1001`，默认按显卡探测值设置 `NUM_ENVS`（若为 **A100 80GB** 可在下方 `else:` 分支手动改写 `NUM_ENVS = 4096`）。

**Colab 中断清空后如何一键断点续训**：
1. 训练期间每次保存检查点（`model_latest.pt` / `latest.ckpt`）及每隔 90 秒，后台都会**实时增量备份**到你的 Google Drive（`MyDrive/SuperMimic_SUGAR_Backup`）；
2. 若 Colab 会话超时被清空，重新连接 GPU 后依次运行 **Step 0 → Step 1 → Step 2 → Step 3 → Step 6.0**（保持相同的 `TASK_NAME` 和 `RUN_MODE`）；
3. Step 6.0 会自动把 Google Drive 中的历史权重、数据集与中间断点拉回 `/content`；随后依次运行 **Stage 6.1 ~ 6.5**，已完成的阶段自动跳过，未完成的阶段自动从上次保存的 Iteration / Epoch 继续往后训！

> **[重要提示：CarryBox 历史权重全部作废]** 本版本针对 `OmniRetarget_Dataset` 更换了参考轨迹、`largebox` 扫描凸包几何、方案二抓取链与 `num_undesired_bodies`（`26 -> 12`）。**此前旧版 `CarryBox` 检查点不可跨版本断点续训**，默认实验名已切换为 `sm_omni_v1_<mode>` 从头训练。

In [ ]:
# ==================== 全局实验配置区 ====================
TASK_NAME = "CarryBox"          # 可选: "CarryBox", "PickBottle", "StandBottle", "PushBox", "KickBox", "SitChair"
RUN_MODE = "full"               # 可选: "full" (5001轮完整训练) | "fast_verify" (快速跑通全流程)
EXP_NAME = f"sm_omni_v1_{RUN_MODE}"  # 实验目录名: outputs/<TASK_NAME>_<EXP_NAME> (使用 v4 隔离全 29 关节残差 + 纯真实回报 Critic 新权重)
SKIP_IF_DONE = True             # 断点续跑保护：若某阶段最终产物已存在，自动跳过；若仅有中间检查点，自动续训
COMPILE_BPTT = True             # 开启单步整图编译 (FK + obs + WM + reward)，消除 Python 循环启动瓶颈
COMPILE_MODE = "default"        # 推荐保持 "default"（秒级 Inductor 融合）；请勿用 "reduce-overhead"（32步BPTT会录制64层CUDA Graph树导致卡顿）
CARRYBOX_MOTION_DIR = "data/OmniCarryBox"  # CarryBox 专属 OmniRetarget 数据集（含 obj_aligned.usda 凸包）
RAW_MOTION_DIR = CARRYBOX_MOTION_DIR if TASK_NAME == "CarryBox" else f"data/{TASK_NAME}"
MAX_FINAL_OBJ_ERR = 0.08 if (TASK_NAME == "CarryBox" and RUN_MODE == "full") else None  # 放置误差过滤阈值 (m)
# ========================================================

if RUN_MODE == "fast_verify":
    NUM_ENVS = min(512, SUGGESTED_FULL_ENVS)
    ROLLOUT_ENVS = min(256, SUGGESTED_FULL_ENVS)
    RL_ITERS = 301
    SAVE_INTERVAL = 50
    IL_EPOCHS = 101
    HORIZON = 32
else:
    NUM_ENVS = SUGGESTED_FULL_ENVS
    ROLLOUT_ENVS = min(1000, SUGGESTED_FULL_ENVS)
    RL_ITERS = 5001
    SAVE_INTERVAL = 200
    IL_EPOCHS = 1001
    HORIZON = 32

LAST_ITER = RL_ITERS - 1
LAST_EPOCH = IL_EPOCHS - 1
USE_TARGET = "True" if TASK_NAME in ("CarryBox", "PickBox", "PushBox") else "False"
IL_WORKERS = min(4, max(2, (os.cpu_count() or 2) - 1))  # 自适应 Colab CPU 核数，防止 DataLoader 过多 worker 卡顿

OUT_DIR = REPO_DIR / "outputs" / f"{TASK_NAME}_{EXP_NAME}"
LOG_ROOT = OUT_DIR / "logs"
CKPT_DIR = OUT_DIR / "ckpts"
LOG_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# 若挂载了 Google Drive 且之前已有备份日志/检查点/数据集，自动恢复到本地以支持单阶段中途续训与跨阶段跳过
if BACKUP_DIR is not None and (BACKUP_DIR / "outputs" / f"{TASK_NAME}_{EXP_NAME}").is_dir():
    subprocess.run(
        ["rsync", "-a", "--exclude=*.tmp", f"{BACKUP_DIR / 'outputs' / f'{TASK_NAME}_{EXP_NAME}'}/", f"{OUT_DIR}/"],
        check=True,
    )
    print(f"[Drive Restore] 已从 Google Drive 恢复历史检查点与 TensorBoard 日志 -> {OUT_DIR}")

REFINER_CKPT = CKPT_DIR / "refiner.pt"
REFINER_RL_DATASET = OUT_DIR / "rollout_datasets" / "refiner" / "rl_dataset"
TRACKER_CKPT = CKPT_DIR / "tracker.pt"
TRACKER_IL_DATASET = OUT_DIR / "rollout_datasets" / "tracker" / "il_dataset"
GENERATOR_CKPT = CKPT_DIR / "generator.ckpt"

# 导出环境变量供下方 %tensorboard 魔法命令精确读取
os.environ["TB_LOGDIR"] = str(LOG_ROOT)
os.environ["TB_SMOKE_LOGDIR"] = str(REPO_DIR / "outputs" / "smoke_test_refiner")

print(f"任务名称         : {TASK_NAME} (use_target={USE_TARGET}) | 参考数据目录: {RAW_MOTION_DIR}")
print(f"运行模式         : {RUN_MODE} | RL_ITERS={RL_ITERS} | IL_EPOCHS={IL_EPOCHS} | NUM_ENVS={NUM_ENVS}")
print(f"TensorBoard 目录 : {LOG_ROOT}")

### 6.0b（强烈推荐）：在训练开始前预启动 Colab 内嵌实时 TensorBoard 面板

- **使用方法**：直接运行下方单元格即可在 Notebook 中嵌入交互式 TensorBoard 窗口（这里先同时挂载刚刚 Step 5 生成的冒烟日志与即将写入的正式实验目录 `LOG_ROOT`，你可以立即看到冒烟曲线确认面板正常；在下方的 Stage 6.1 / 6.3 / 6.5 运行期间，点击 TensorBoard 右上角的 **刷新按钮（Reload data）** 或在设置齿轮中勾选 `Reload data (30s)` 即可边跑边看实时曲线！）。

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir {LOG_ROOT}

### Stage 6.1：训练 Stage 1 `SuperMimic Refiner`（替换原版 RSL-RL PPO）

- **作用**：在原始参考动作 `data/<TASK_NAME>` 上，利用人-物联合世界模型 `JointWorldModel` 与 `H=32` 步想象展开 BPTT 训练 `890` 维特权观测专家策略 `Refiner`（TensorBoard 实时写入 `logs/refiner/`）。
- **预期结果**：导出 `outputs/<TASK>_<EXP>/ckpts/refiner.pt` 并自动同步备份至 Google Drive。

In [ ]:
if SKIP_IF_DONE and REFINER_CKPT.is_file():
    print(f"[跳过 Stage 6.1] 已存在最终 Refiner 权重: {REFINER_CKPT}")
else:
    refiner_log_dir = LOG_ROOT / "refiner"
    expected_model = refiner_log_dir / f"model_{LAST_ITER}.pt"
    if (refiner_log_dir / "model_latest.pt").is_file() or any(refiner_log_dir.glob("model_*.pt")):
        print(f"[断点续训 Stage 6.1] 检测到中间检查点 ({refiner_log_dir})，将自动从上次中断轮数继续训练！")
    compile_args = ["--compile_bptt", "--compile_mode", COMPILE_MODE] if COMPILE_BPTT else []
    run_training_stage(
        [
            PYTHON_BIN, "-u", "scripts/supermimic/train.py",
            "--task", f"Sugar-G129dof-{TASK_NAME}-Refiner",
            "--num_envs", str(NUM_ENVS),
            "--horizon", str(HORIZON),
            "--max_iterations", str(RL_ITERS),
            "--save_interval", str(SAVE_INTERVAL),
            "--motion_folder", RAW_MOTION_DIR,
            "--log_dir", str(refiner_log_dir),
            "--auto_resume",
            *compile_args,
            "--headless",
        ],
        env=BASE_ENV,
        log_file=refiner_log_dir / "train_refiner.log",
        allow_isaac_139=True,
        expected_output=expected_model,
        sync_rel_paths=[f"outputs/{TASK_NAME}_{EXP_NAME}/logs/refiner"],
    )
    shutil.copy2(expected_model, REFINER_CKPT)
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/ckpts")
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/logs/refiner")
    print(f"[Stage 6.1 完成] 已导出 SuperMimic Refiner -> {REFINER_CKPT}")

### Stage 6.2：`SuperMimic Refiner` 物理轨迹回放与可行数据集构建

- **作用**：回放 `refiner.pt` 采集真实物理轨迹，并由 `process_refiner_rollout.py` 转换为供 Tracker 训练的 `rl_dataset`。
- **预期结果**：生成非空的 `outputs/<TASK>_<EXP>/rollout_datasets/refiner/rl_dataset` 目录。

In [ ]:
if SKIP_IF_DONE and REFINER_RL_DATASET.is_dir() and any(REFINER_RL_DATASET.rglob("*.npz")):
    print(f"[跳过 Stage 6.2] 已存在精炼数据集: {REFINER_RL_DATASET}")
else:
    raw_refiner_dir = OUT_DIR / "rollout_datasets" / "refiner" / "raw_npz"
    raw_refiner_dir.mkdir(parents=True, exist_ok=True)
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/supermimic/play.py",
            "--task", f"Sugar-G129dof-{TASK_NAME}-Refiner-Rollout",
            "--num_envs", str(ROLLOUT_ENVS),
            "--checkpoint", str(REFINER_CKPT),
            "--rollout_dir", str(raw_refiner_dir),
            "--motion_folder", RAW_MOTION_DIR,
            "--headless",
        ],
        env=BASE_ENV,
        log_file=LOG_ROOT / "refiner_rollout.log",
        allow_isaac_139=True,
        expected_output=raw_refiner_dir,
    )
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/supermimic/analyze_rollouts.py",
            "--rollout_dir", str(raw_refiner_dir),
            "--ref_dir", RAW_MOTION_DIR,
            "--ckpt", str(REFINER_CKPT),
            "--out_json", str(LOG_ROOT / "refiner_rollout_metrics.json"),
        ],
        env=BASE_ENV,
    )
    filter_args = ["--max_final_obj_err", str(MAX_FINAL_OBJ_ERR)] if MAX_FINAL_OBJ_ERR is not None else []
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/sugar_rl/process_refiner_rollout.py",
            "--data_dir", str(OUT_DIR / "rollout_datasets" / "refiner"),
            "--task_name", TASK_NAME,
            *filter_args,
        ],
        env=BASE_ENV,
    )
    refined_count = len(list(REFINER_RL_DATASET.rglob("*.npz")))
    assert refined_count > 0, (
        "Refiner Rollout 过滤后得到 0 条完整轨迹！若当前为 fast_verify 轮数较少导致早期成功率偏低，可将 RL_ITERS 调至 800~1500 后重跑 Stage 6.1。"
    )
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/rollout_datasets/refiner/rl_dataset")
    print(f"[Stage 6.2 完成] 物理精炼轨迹数: {refined_count} -> {REFINER_RL_DATASET}")

### Stage 6.3：训练 Stage 2 `SuperMimic Tracker`（32 步 WM-BPTT + 冻结 Teacher BC 蒸馏）

- **作用**：训练仅依赖 `510` 维观测（36 维单帧指令 + 5 帧滚动本体历史）的可部署全身控制器 `SuperMimic Tracker`，同时接收 32 步世界模型 BPTT 梯度与冻结 `refiner.pt` 的动态退火蒸馏损失（TensorBoard 实时写入 `logs/tracker/`）。
- **预期结果**：导出 `outputs/<TASK>_<EXP>/ckpts/tracker.pt` 并自动备份至 Google Drive。

In [ ]:
if SKIP_IF_DONE and TRACKER_CKPT.is_file():
    print(f"[跳过 Stage 6.3] 已存在最终 Tracker 权重: {TRACKER_CKPT}")
else:
    tracker_log_dir = LOG_ROOT / "tracker"
    expected_tracker = tracker_log_dir / f"model_{LAST_ITER}.pt"
    if (tracker_log_dir / "model_latest.pt").is_file() or any(tracker_log_dir.glob("model_*.pt")):
        print(f"[断点续训 Stage 6.3] 检测到中间检查点 ({tracker_log_dir})，将自动从上次中断轮数继续训练！")
    compile_args = ["--compile_bptt", "--compile_mode", COMPILE_MODE] if COMPILE_BPTT else []
    run_training_stage(
        [
            PYTHON_BIN, "-u", "scripts/supermimic/train.py",
            "--task", f"Sugar-G129dof-{TASK_NAME}-Tracker",
            "--num_envs", str(NUM_ENVS),
            "--horizon", str(HORIZON),
            "--teacher_ckpt", str(REFINER_CKPT),
            "--motion_folder", str(REFINER_RL_DATASET),
            "--teacher_motion_folder", RAW_MOTION_DIR,
            "--max_iterations", str(RL_ITERS),
            "--save_interval", str(SAVE_INTERVAL),
            "--log_dir", str(tracker_log_dir),
            "--auto_resume",
            *compile_args,
            "--headless",
        ],
        env=BASE_ENV,
        log_file=tracker_log_dir / "train_tracker.log",
        allow_isaac_139=True,
        expected_output=expected_tracker,
        sync_rel_paths=[f"outputs/{TASK_NAME}_{EXP_NAME}/logs/tracker"],
    )
    shutil.copy2(expected_tracker, TRACKER_CKPT)
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/ckpts")
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/logs/tracker")
    print(f"[Stage 6.3 完成] 已导出 SuperMimic Tracker -> {TRACKER_CKPT}")

### Stage 6.4：`SuperMimic Tracker` 闭环轨迹采集与模仿学习 Zarr 数据集转换

- **作用**：采集 `SuperMimic Tracker` 的真实闭环执行轨迹，并通过 `process_tracker_rollout.py` 构建 10 Hz Zarr 数据集 `il_dataset`。
- **预期结果**：生成 `outputs/<TASK>_<EXP>/rollout_datasets/tracker/il_dataset`。

In [ ]:
if SKIP_IF_DONE and (TRACKER_IL_DATASET / ".zgroup").is_file():
    print(f"[跳过 Stage 6.4] 已存在模仿学习 Zarr 数据集: {TRACKER_IL_DATASET}")
else:
    raw_tracker_dir = OUT_DIR / "rollout_datasets" / "tracker" / "raw_npz"
    raw_tracker_dir.mkdir(parents=True, exist_ok=True)
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/supermimic/play.py",
            "--task", f"Sugar-G129dof-{TASK_NAME}-Tracker-Rollout",
            "--checkpoint", str(TRACKER_CKPT),
            "--num_envs", str(ROLLOUT_ENVS),
            "--rollout_dir", str(raw_tracker_dir),
            "--motion_folder", str(REFINER_RL_DATASET),
            "--teacher_motion_folder", RAW_MOTION_DIR,
            "--teacher_ckpt", str(REFINER_CKPT),
            "--headless",
        ],
        env=BASE_ENV,
        log_file=LOG_ROOT / "tracker_rollout.log",
        allow_isaac_139=True,
        expected_output=raw_tracker_dir,
    )
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/sugar_rl/process_tracker_rollout.py",
            "--data_dir", str(OUT_DIR / "rollout_datasets" / "tracker"),
        ],
        env=BASE_ENV,
    )
    assert TRACKER_IL_DATASET.is_dir(), f"未找到转换后的 Zarr 数据集: {TRACKER_IL_DATASET}"
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/rollout_datasets/tracker/il_dataset")
    print(f"[Stage 6.4 完成] 已生成模仿学习 Zarr 数据集 -> {TRACKER_IL_DATASET}")

### Stage 6.5：训练高层扩散指令生成器 `Diffusion Generator`（`sugar_il`）

- **作用**：在 `il_dataset` 上训练扩散策略 `Generator`，学习根据当前机器人与物体位姿实时生成 36 维全身+接触指令（`Accelerate` 自动将训练曲线写入 `logs/generator/tb/train_generator/`，且自动适配 A100/L4 `bf16` 与 T4 `fp16`）。
- **预期结果**：导出 `outputs/<TASK>_<EXP>/ckpts/generator.ckpt` 并自动备份至 Google Drive。

In [ ]:
if SKIP_IF_DONE and GENERATOR_CKPT.is_file():
    print(f"[跳过 Stage 6.5] 已存在最终 Generator 权重: {GENERATOR_CKPT}")
else:
    gen_log_dir = LOG_ROOT / "generator"
    expected_gen_epoch = gen_log_dir / "epoch_checkpoints" / f"epoch={LAST_EPOCH}.ckpt"
    if (gen_log_dir / "checkpoints" / "latest.ckpt").is_file() or any((gen_log_dir / "epoch_checkpoints").glob("epoch=*.ckpt")):
        print(f"[断点续训 Stage 6.5] 检测到中间检查点 ({gen_log_dir})，将自动从上次中断 Epoch 继续训练！")
    run_training_stage(
        [
            PYTHON_BIN, "-u", "scripts/sugar_il/train.py",
            "--config-name", "train_generator_workspace.yaml",
            f"task={TASK_NAME}",
            f"use_target={USE_TARGET}",
            f"num_epochs={IL_EPOCHS}",
            "training.resume=True",
            "checkpoint.save_last_ckpt=True",
            f"dataloader.num_workers={IL_WORKERS}",
            f"val_dataloader.num_workers={IL_WORKERS}",
            f"log_path={gen_log_dir}",
            f"dataset_path={TRACKER_IL_DATASET}",
        ],
        env=BASE_ENV,
        log_file=LOG_ROOT / "train_generator.log",
        expected_output=expected_gen_epoch,
        sync_rel_paths=[f"outputs/{TASK_NAME}_{EXP_NAME}/logs/generator"],
    )
    epoch_ckpts = sorted((gen_log_dir / "epoch_checkpoints").glob("epoch=*.ckpt"), key=lambda p: p.stat().st_mtime)
    assert epoch_ckpts, f"未在 {gen_log_dir / 'epoch_checkpoints'} 下找到任何 .ckpt 文件！"
    shutil.copy2(epoch_ckpts[-1], GENERATOR_CKPT)
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/ckpts")
    sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/logs/generator")
    print(f"[Stage 6.5 完成] 已导出 Diffusion Generator -> {GENERATOR_CKPT}")

---
## Step 7：50 Hz 闭环推理 / 阶段模型导出 MP4 视频并同步 Google Drive

- **本步作用**：在 Isaac Sim 真实物理仿真中运行策略并导出 `.mp4` 视频（同步保存至 Google Drive，且在 Colab 下自动内嵌播放）。
- **Colab 兼容性说明**：Google Colab 容器宿主机未提供 Vulkan 1.1 图形驱动，无法启动 Isaac Sim 自带的 `--video` 渲染器；因此本单元格在 Colab 下自动采用 **「Isaac Sim 纯物理仿真 + MuJoCo `osmesa` 离屏渲染」**（物理 `100%` 跑在 Isaac Sim CUDA PhysX 上，仅借用 CPU 离屏绘制 3D 画面），无需任何本地下载或额外配置即可直接出 MP4。
- **模式切换（`PREVIEW_STAGE`）**：
  - `"inference"`（默认）：加载 `tracker.pt` + `generator.ckpt` 全自主闭环推理（也可将 `EVAL_GEN_CKPT` 临时改为 `REPO_DIR / "demo_ckpts" / TASK_NAME / "generator.ckpt"`）；
  - `"refiner"` / `"tracker"`：只训完 Stage 6.1 或 Stage 6.3 时，也可随时切换单独预览当前策略在 Isaac Sim 中的表现。

In [ ]:
from IPython.display import Video, display

PREVIEW_STAGE = "inference"   # 可选: "inference" (闭环) | "refiner" (仅预览Stage 6.1) | "tracker" (仅预览Stage 6.3)
EVAL_TRACKER_CKPT = TRACKER_CKPT
EVAL_GEN_CKPT = GENERATOR_CKPT
NUM_TRAJS = 3                 # 每次连续生成 3 段轨迹的 MP4 视频
VIDEO_STEPS = 700             # 仅 inference 模式使用的步数上限；refiner/tracker 自动按参考轨迹全帧数录完
RENDER_STRIDE = 2             # 抽帧步长: 2 = 25fps (流畅且 CPU 渲染提速 2 倍；设 1 则为 50fps)
RENDER_WIDTH = 854            # 480p 分辨率 (相比 720p 渲染提速 ~2.25 倍)
RENDER_HEIGHT = 480
USE_ISAAC_NATIVE_VIDEO = not IS_COLAB  # 有完整 Vulkan 驱动的 Linux 桌面/服务器可改为 True 使用原生 --video

video_out_dir = CKPT_DIR / "videos" / "play"
video_out_dir.mkdir(parents=True, exist_ok=True)

if USE_ISAAC_NATIVE_VIDEO and PREVIEW_STAGE == "inference":
    assert EVAL_TRACKER_CKPT.is_file(), f"未找到 Tracker 权重: {EVAL_TRACKER_CKPT}"
    assert EVAL_GEN_CKPT.is_file(), f"未找到 Generator 权重: {EVAL_GEN_CKPT}"
    run_stream(
        [
            PYTHON_BIN, "-u", "scripts/supermimic/play.py",
            "--task", f"Sugar-G129dof-{TASK_NAME}-Inference",
            "--checkpoint", str(EVAL_TRACKER_CKPT),
            "--generator_checkpoint", str(EVAL_GEN_CKPT),
            "--motion_folder", RAW_MOTION_DIR,
            "--num_envs", "4",
            "--eval_random_motion",
            "--headless",
            "--video",
            "--video_length", str(VIDEO_STEPS),
        ],
        env=BASE_ENV,
        log_file=LOG_ROOT / "inference_eval.log",
        allow_isaac_139=True,
        expected_output=video_out_dir,
    )
    mp4_files = sorted(video_out_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    assert mp4_files, f"未在 {video_out_dir} 下找到录制的 MP4 视频！"
    generated_mp4s = mp4_files[-NUM_TRAJS:]
else:
    if PREVIEW_STAGE == "refiner":
        eval_ckpt = REFINER_CKPT
        eval_task_id = f"Sugar-G129dof-{TASK_NAME}-Refiner"
        eval_motion = RAW_MOTION_DIR
        eval_gen = ""
    elif PREVIEW_STAGE == "tracker":
        eval_ckpt = EVAL_TRACKER_CKPT
        eval_task_id = f"Sugar-G129dof-{TASK_NAME}-Tracker"
        eval_motion = str(REFINER_RL_DATASET) if REFINER_RL_DATASET.is_dir() else RAW_MOTION_DIR
        eval_gen = ""
    else:
        eval_ckpt = EVAL_TRACKER_CKPT
        eval_task_id = f"Sugar-G129dof-{TASK_NAME}-Inference"
        eval_motion = RAW_MOTION_DIR
        eval_gen = str(EVAL_GEN_CKPT)
        assert EVAL_GEN_CKPT.is_file(), f"未找到 Generator 权重: {EVAL_GEN_CKPT}"

    assert eval_ckpt.is_file(), f"未找到策略权重: {eval_ckpt}"
    generated_mp4s = [video_out_dir / f"{TASK_NAME}_{PREVIEW_STAGE}_traj{i}.mp4" for i in range(1, NUM_TRAJS + 1)]

    render_isaac_mp4_script = r'''
import os, sys
os.environ["MUJOCO_GL"] = "osmesa"
sys.path.insert(0, os.getcwd())

import torch, gymnasium as gym, mujoco, imageio
from isaaclab.app import AppLauncher
app_launcher = AppLauncher({"headless": True, "device": "cuda:0"})
simulation_app = app_launcher.app

import sugar_rl.tasks
from sugar_rl.utils.parser_cfg import parse_env_cfg
from sugar_rl.supermimic.trainer import load_supermimic_policy
from sugar_rl.supermimic.env_bridge import SugarEnvBridge
from scripts.sim2sim.scene_builder import build_mujoco_scene_xml

task_name, task_id, ckpt_path, motion_dir, gen_ckpt, out_dir, prefix, n_trajs, n_steps, stride, width, height = (
    sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5],
    sys.argv[6], sys.argv[7], int(sys.argv[8]), int(sys.argv[9]), int(sys.argv[10]), int(sys.argv[11]), int(sys.argv[12])
)

is_inf = task_id.endswith("-Inference")
entry_key = "play_env_cfg_entry_point" if is_inf else "env_cfg_entry_point"
env_cfg = parse_env_cfg(task_id, device="cuda:0", num_envs=1, entry_point_key=entry_key)
env_cfg.episode_length_s = 120.0
if hasattr(env_cfg, "events"):
    if hasattr(env_cfg.events, "push_robot"):
        env_cfg.events.push_robot = None
    if hasattr(env_cfg.events, "push_object"):
        env_cfg.events.push_object = None
if hasattr(env_cfg, "commands") and hasattr(env_cfg.commands, "motion"):
    m_cfg = env_cfg.commands.motion
    m_cfg.motion_folder = motion_dir
    from sugar_rl.assets.objects.objects import apply_dataset_box_spec
    apply_dataset_box_spec(env_cfg, motion_dir)
    if gen_ckpt and hasattr(m_cfg, "generator_checkpoint_path"):
        m_cfg.generator_checkpoint_path = gen_ckpt
    if hasattr(m_cfg, "eval_random_motion"):
        m_cfg.eval_random_motion = True
    if not is_inf:
        m_cfg.pose_range = {k: (0.0, 0.0) for k in ("x", "y", "z", "roll", "pitch", "yaw")}
        m_cfg.joint_position_range = (0.0, 0.0)

env = gym.make(task_id, cfg=env_cfg)
bridge = SugarEnvBridge(env=env, task_name_or_id=task_id, verify_obs_on_init=False)
policy = load_supermimic_policy(ckpt_path, device="cuda:0")
joint_names = list(bridge.fk.env_joint_names)

xml_path, _, _ = build_mujoco_scene_xml(task_name, motion_folder=motion_dir)
mj_model = mujoco.MjModel.from_xml_path(str(xml_path))
mj_data = mujoco.MjData(mj_model)
p_adr = int(mj_model.jnt_qposadr[mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_JOINT, "pelvis_freejoint")])
o_adr = int(mj_model.jnt_qposadr[mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_JOINT, "obj_freejoint")])
j_adrs = [int(mj_model.jnt_qposadr[mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_JOINT, n)]) for n in joint_names]
t_body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "target_vis")
t_mocap_id = int(mj_model.body_mocapid[t_body_id]) if t_body_id >= 0 else -1

cam = mujoco.MjvCamera()
cam.distance = 2.8
cam.elevation = -18.0
cam.azimuth = 135.0
renderer = mujoco.Renderer(mj_model, height=height, width=width)
fps_out = max(1, 50 // max(1, stride))
os.makedirs(out_dir, exist_ok=True)

for traj_idx in range(1, n_trajs + 1):
    obs_dict, _ = env.reset()
    bridge.current_obs_dict = obs_dict
    origin = bridge.env.scene.env_origins[0]
    goal_pos = (bridge.motion_cmd.obj_target_pos_w[0] - origin).detach().cpu().numpy()
    goal_quat = bridge.motion_cmd.obj_target_quat_w[0].detach().cpu().numpy()
    m_id = int(bridge.motion_cmd.motion_id[0].item())
    total_frames = int(bridge.motion_cmd.motion.time_step_total_permotion[m_id].item()) if not is_inf else n_steps
    traj_s = []
    status = "max_steps"
    step_idx = 0
    try:
        while step_idx < ((total_frames + 10) if not is_inf else n_steps):
            with torch.no_grad():
                act = policy.act_inference(obs_dict["policy"])
            obs_dict, _, dones, timeouts, _, _ = bridge.step(act)
            if not is_inf:
                if step_idx <= 1 and bool(dones[0].item()):
                    traj_s.clear()
                    origin = bridge.env.scene.env_origins[0]
    goal_pos = (bridge.motion_cmd.obj_target_pos_w[0] - origin).detach().cpu().numpy()
    goal_quat = bridge.motion_cmd.obj_target_quat_w[0].detach().cpu().numpy()
    m_id = int(bridge.motion_cmd.motion_id[0].item())
                    total_frames = int(bridge.motion_cmd.motion.time_step_total_permotion[m_id].item())
                    step_idx += 1
                    continue
                if bool(dones[0].item()):
                    status = "trajectory_complete" if bool(timeouts[0].item()) else "terminated_early"
                    break
            traj_s.append(bridge.get_joint_state()[0].detach().cpu().numpy())
            step_idx += 1
            if not is_inf and len(traj_s) >= total_frames:
                status = "trajectory_complete"
                break
    except SystemExit:
        status = "system_exit"

    sub_traj = traj_s[::max(1, stride)]
    frames = []
    for f_i, s in enumerate(sub_traj, 1):
        mj_data.qpos[p_adr:p_adr+3] = s[0:3]
        mj_data.qpos[p_adr+3:p_adr+7] = s[3:7]
        mj_data.qpos[j_adrs] = s[7:36]
        mj_data.qpos[o_adr:o_adr+3] = s[71:74]
        mj_data.qpos[o_adr+3:o_adr+7] = s[74:78]
        if t_mocap_id >= 0:
            mj_data.mocap_pos[t_mocap_id] = goal_pos
            mj_data.mocap_quat[t_mocap_id] = goal_quat
        mujoco.mj_forward(mj_model, mj_data)
        cam.lookat[:] = [float(s[0]), float(s[1]), 0.85]
        renderer.update_scene(mj_data, camera=cam)
        frames.append(renderer.render())
        if f_i % 100 == 0 or f_i == len(sub_traj):
            print(f"  [Traj {traj_idx}/{n_trajs}] 渲染 {f_i}/{len(sub_traj)} 帧", flush=True)

    out_mp4 = os.path.join(out_dir, f"{prefix}_traj{traj_idx}.mp4")
    imageio.mimsave(out_mp4, frames, fps=fps_out)
    print(f"[OK] 轨迹 {traj_idx}/{n_trajs} ({status}, 物理 {len(traj_s)} 步 -> {len(frames)} 帧@{fps_out}fps) -> {out_mp4}", flush=True)

renderer.close()
env.close()
simulation_app.close()
'''

    run_stream(
        [
            PYTHON_BIN, "-u", "-c", render_isaac_mp4_script,
            TASK_NAME, eval_task_id, str(eval_ckpt), eval_motion, eval_gen,
            str(video_out_dir), f"{TASK_NAME}_{PREVIEW_STAGE}",
            str(NUM_TRAJS), str(VIDEO_STEPS), str(RENDER_STRIDE), str(RENDER_WIDTH), str(RENDER_HEIGHT),
        ],
        env=BASE_ENV,
        log_file=LOG_ROOT / f"{PREVIEW_STAGE}_eval.log",
        allow_isaac_139=True,
        expected_output=generated_mp4s[-1],
    )

sync_to_drive(f"outputs/{TASK_NAME}_{EXP_NAME}/ckpts/videos")
for mp4_p in generated_mp4s:
    assert mp4_p.is_file(), f"未找到导出的 MP4 视频: {mp4_p}"
    print(f"\n[导出完成] 视频文件: {mp4_p}")
    display(Video(str(mp4_p), embed=True, width=760))

---
## Step 8：全方位查看训练曲线（Colab 内嵌 TensorBoard + 离线高清图表 + 一键下载）

全部三个训练阶段均已写入标准的 TensorBoard Event 文件：
- **Stage 1 (`refiner`) & Stage 3 (`tracker`) 核心指标**：
  - `collect/sim_step_reward` / `collect/reset_rate`：Isaac Sim 真实物理环境每步平均奖励与重置率；
  - `policy/imagined_return` / `policy/tail_survival_weight` / `policy/imagined_obj_rew`：32 步可微世界模型想象展开累积回报、尾部存活权重与物体交互子奖励；
  - `wm/pos_err_mm` / `wm/obj_err_mm` / `wm/contact_bce`：人-物联合世界模型在 8 步自回归展开下的机器人位姿误差（mm）、物体位姿误差（mm）与接触预测 BCE 损失；
  - `policy/bc_loss` / `policy/bc_coef`（仅 Tracker）：冻结 Stage 1 Refiner 动作蒸馏误差与退火系数；
  - `critic/loss` / `perf/fps`：多头价值网络损失与每秒采样帧率。
- **Stage 5 (`generator/tb/train_generator`) 核心指标**：
  - `train_loss` / `train_action_mse_error` / `val_action_mse_error` / `lr`：扩散策略噪声预测损失与 36 维指令重构 MSE。

### 8.1 方式一：启动交互式 TensorBoard 面板

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir {LOG_ROOT}

### 8.2 方式二：直接解析 TensorBoard `tfevents` 绘制高清训练曲线汇总图（免 Cookie 白屏困扰）

> 如果你的浏览器开启了无痕模式或严格阻止第三方 Cookie，Colab 内嵌的 `%tensorboard` iframe 可能会显示空白。运行下方单元格将直接读取磁盘上的 `events.out.tfevents.*` 文件，打印所有已记录阶段的精确指标摘要，并绘制高清 PNG 汇总曲线（同时支持一键打包下载全部 TensorBoard 日志回本地查看）！

In [ ]:
from IPython.display import Image, display

plot_script = """
import os, sys, zipfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_root = Path(sys.argv[1])
smoke_dir = Path(sys.argv[2])
out_png = log_root / "tensorboard_summary_curves.png"

search_roots = [log_root]
if not list(log_root.rglob("events.out.tfevents.*")) and smoke_dir.is_dir():
    search_roots.append(smoke_dir)

event_dirs = sorted({p.parent for r in search_roots for p in r.rglob("events.out.tfevents.*")})
if not event_dirs:
    print("尚未找到任何 events.out.tfevents.* 文件，请先运行 Step 5 或 Step 6 训练阶段。")
    sys.exit(0)

runs = {}
print("=== 已检测到的 TensorBoard 运行目录与标量指标 ===")
for d in event_dirs:
    ea = EventAccumulator(str(d))
    ea.Reload()
    tags = ea.Tags().get("scalars", [])
    if not tags:
        continue
    rel_name = str(d.relative_to(log_root)) if d.is_relative_to(log_root) else d.name
    runs[rel_name] = ea
    sample_info = ", ".join(f"{t}({len(ea.Scalars(t))}pts, last={ea.Scalars(t)[-1].value:.4f})" for t in tags[:5])
    print(f"  [{rel_name}] 共 {len(tags)} 个标量指标 -> {sample_info} ...")

panels = [
    ("Real Sim Step Reward", ["collect/sim_step_reward"]),
    ("32-Step Imagined Return", ["policy/imagined_return"]),
    ("World Model Pos Error (mm)", ["wm/pos_err_mm", "wm/obj_err_mm"]),
    ("Critic & Contact Loss", ["critic/loss", "wm/contact_bce"]),
    ("Tracker Teacher BC Loss", ["policy/bc_loss"]),
    ("Diffusion Generator Loss", ["train_loss", "train_action_mse_error"]),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), dpi=130)
for ax, (title, candidate_tags) in zip(axes.flat, panels):
    plotted = False
    for run_name, ea in runs.items():
        avail = set(ea.Tags().get("scalars", []))
        for tag in candidate_tags:
            if tag in avail:
                pts = ea.Scalars(tag)
                steps = [p.step for p in pts]
                vals = [p.value for p in pts]
                ax.plot(steps, vals, linewidth=1.8, label=f"{run_name}:{tag}")
                plotted = True
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Iteration / Step")
    ax.grid(True, alpha=0.3)
    if plotted:
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "Pending stage execution", ha="center", va="center", transform=ax.transAxes, color="gray")

plt.tight_layout()
out_png.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_png)
print(f"\n[已生成汇总曲线图] {out_png}")
"""

run_stream([PYTHON_BIN, "-c", plot_script, str(LOG_ROOT), str(REPO_DIR / "outputs" / "smoke_test_refiner")], env=BASE_ENV)

summary_png = LOG_ROOT / "tensorboard_summary_curves.png"
if summary_png.is_file():
    display(Image(filename=str(summary_png)))

# 可选：若需要将全部 TensorBoard 日志打包下载到本地电脑查看，将 DOWNLOAD_TB_ZIP 设为 True
DOWNLOAD_TB_ZIP = False
if IS_COLAB and DOWNLOAD_TB_ZIP:
    tb_zip = REPO_DIR / "outputs" / f"{TASK_NAME}_{EXP_NAME}_tensorboard_logs.zip"
    with zipfile.ZipFile(tb_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in LOG_ROOT.rglob("events.out.tfevents.*"):
            zf.write(p, p.relative_to(LOG_ROOT))
    from google.colab import files  # type: ignore
    files.download(str(tb_zip))